In [3]:
from db_config import run_query

ALLOWED = {"orgc": "wosis_orgc", 
           "ph": "wosis_phaq", 
           "clay": "wosis_clay", 
           "sand": "wosis_sand"}


def count_rows(short_name, country="Ghana"):
    if short_name not in ALLOWED:
        raise ValueError(f"Unknown table: {short_name}")
    table = ALLOWED[short_name]
    query = f"SELECT COUNT(*) AS n FROM {table} WHERE country_name = '{country}';"
    return run_query(query)["n"].iloc[0]


def build_flat(base, others, country="Ghana"):
    # 1. Check all names are safe
    if base not in ALLOWED:
        raise ValueError(f"Unknown table: {base}")
    for name in others:
        if name not in ALLOWED:
            raise ValueError(f"Unknown table: {name}")

    # 2. SELECT parts
    select_parts = [
        "b.profile_id",
        "b.layer_id",
        f"b.value_avg AS {base}"
    ]

    # 3. JOIN parts
    join_parts = []
    for i, name in enumerate(others, start=1):
        alias = f"t{i}"
        table = ALLOWED[name]
        select_parts.append(f"{alias}.value_avg AS {name}")
        join_parts.append(
            f"LEFT JOIN {table} AS {alias} ON b.layer_id = {alias}.layer_id"
        )

    # 4. Whole query
    query = f"""
    SELECT {', '.join(select_parts)}
    FROM {ALLOWED[base]} AS b
    {' '.join(join_parts)}
    WHERE b.country_name = '{country}';
    """

    # 5. Run it
    df = run_query(query)

    # 6. Check against the same country
    expected = count_rows(base, country=country)
    if expected == 0:
            raise ValueError(f"No data for country: {country}")

    if len(df) != expected:
        raise ValueError(
            f"Row count changed: got {len(df)}, expected {expected}"
        )

    # 7. Return safe DataFrame
    return df

In [4]:
flat_org = build_flat("orgc", ["ph","clay","sand"])

In [5]:
print(flat_org.shape)

(1756, 6)


In [6]:
flat_sand = build_flat("sand", ["clay","ph"])

In [7]:
flat_sand.shape

(834, 5)

In [8]:
build_flat("orgc", ["ph", "clya"])

ValueError: Unknown table: clya

In [9]:
flat_org["orgc"].sum()

np.float64(123653.40000000001)

In [ ]:
from db_config import get_engine
engine = get_engine()

flat = build_flat("orgc", ["ph", "clay", "sand"])
flat.to_sql("flat_orgc_ghana", engine, if_exists="replace", index=False)
check = run_query("SELECT * FROM flat_orgc_ghana;")

In [11]:
flat.shape == check.shape

True

In [12]:
flat.duplicated().sum()

np.int64(0)

In [13]:
flat["layer_id"].is_unique

True

In [14]:
checked = flat.copy()

checked["flag_ph"]        = (checked['ph'] < 0)|(checked['ph'] > 14)
checked["flag_clay_sand"] = (checked['sand']+ checked['clay']) > 100
checked["flag_clay"]      = (checked['clay'] < 0)|(checked['clay']> 100)
checked["flag_sand"]      = (checked['sand'] < 0)|(checked['sand']> 100)
checked["flag_oc_neg"]    = checked['orgc'] < 0

In [15]:
flag_cols = checked[['flag_ph', 'flag_clay_sand', 'flag_clay', 'flag_sand', 'flag_oc_neg']]
print(flag_cols.sum())
print(flag_cols.any(axis=1).sum())


flag_ph           0
flag_clay_sand    0
flag_clay         0
flag_sand         0
flag_oc_neg       0
dtype: int64
0


In [16]:
test = flat.copy()
test.loc[test.index[0], "ph"] = 14.9
((test["ph"] < 0) | (test["ph"] > 14)).sum()   # expect 1

np.int64(1)

In [17]:
flat[["orgc", "ph", "clay", "sand"]].describe()

,orgc,ph,clay,sand
count,1756.000000,1742.000000,499.000000,500.000000
mean,70.417654,5.859644,30.021042,48.416400
std,87.999765,1.133373,16.447999,20.172201
min,0.100000,3.800000,2.000000,4.000000
25%,4.400000,5.000000,15.550000,34.800000
50%,35.800000,5.600000,31.000000,46.000000
75%,108.500000,6.600000,43.000000,60.200000
max,638.700000,8.900000,72.000000,93.100000
